<a href="https://colab.research.google.com/github/mdaminu2002-sketch/bank_fraud/blob/main/fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install --no-deps unsloth
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets trl

  Using cached xformers-0.0.26.post1.tar.gz (4.1 MB)
  Preparing metadata (setup.py) ... done
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for xformers
  Running setup.py clean for xformers
Failed to build xformers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (xformers)
  Using cached trl-1.9.0-py3-none-any.whl.metadata (12 kB)
  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached pyarrow-25.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.me

In [3]:
import torch
import json
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

# ----------------------------------------------------
# A. Get 2,000 Hausa Pairs & Format Dataset
# ----------------------------------------------------
print("1/3 Downloading and formatting Hausa dataset...")
raw_dataset = load_dataset("michsethowusu/english-hausa_sentence-pairs_mt560", split="train[:2000]")

formatted_data = []
for item in raw_dataset:
    en_text = item.get("english", "")
    ha_text = item.get("hausa", "")
    if en_text and ha_text:
        formatted_data.append({
            "messages": [
                {"role": "user", "content": f"Translate to Hausa or respond in Hausa:\n{en_text}"},
                {"role": "model", "content": ha_text}
            ]
        })

# Save locally inside Colab
with open("hausa_dataset.json", "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, ensure_ascii=False, indent=2)

print("Saved hausa_dataset.json inside Colab!")

# ----------------------------------------------------
# B. Load Base Gemma Model (4-bit QLoRA for Fast VRAM)
# ----------------------------------------------------
print("2/3 Loading Gemma model into Colab GPU...")
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it", # Extremely fast & lightweight for free Colab GPU
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# ----------------------------------------------------
# C. Add LoRA Adapters
# ----------------------------------------------------
print("3/3 Setting up LoRA training adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

print("Setup Complete! Ready for Fine-Tuning.")

ImportError: Unsloth: Please install unsloth_zoo via `pip install unsloth_zoo` then retry!